# 02 · Reset credentials — Calendar Agent

Use this when the bot reports an authentication error, when a credential expires, or after a secret leaks. The notebook is split by credential type so you only do the part you need.

The bot uses 4 kinds of credentials, and each fails in a different way.

| Credential | Lives in | Symptom | Section |
|---|---|---|---|
| Google Calendar token | `data/token.json` | Bot says "auth error / connection error"; log shows `invalid_grant` | **A** |
| Google OAuth client | `data/credentials.json` | `auth.py` fails with a client error / `invalid_client`, or the file is missing | **B** |
| Discord bot token | `.env` → `DISCORD_BOT_TOKEN` | Bot never comes online; log shows `LoginFailure` / `Improper token` | **C** |
| Claude API key | `.env` → `ANTHROPIC_API_KEY` | Bot reports `401`, `credit balance` or `billing` | **D** |

Commands in Markdown cells run in **PowerShell** from `C:\Calendar_Agent`. Python code cells are diagnostics/verification and never print secrets.

## 0. Diagnose: which credential is broken?

Run the cell below. It scans `logs/bot.log` and tells you which kind of failure appeared most recently.

In [ ]:
from pathlib import Path

# Locate the project root (the folder containing docker-compose.yml),
# whether this notebook is opened from docs/ or from the project root.
ROOT = Path.cwd()
if not (ROOT / "docker-compose.yml").exists() and (ROOT.parent / "docker-compose.yml").exists():
    ROOT = ROOT.parent
print("Project folder:", ROOT)
print("docker-compose.yml found:", (ROOT / "docker-compose.yml").exists())

import re, time

log = ROOT / "logs" / "bot.log"
if not log.exists():
    print("No logs/bot.log found")
else:
    text = log.read_text(encoding="utf-8", errors="ignore")
    checks = {
        "A - Google token expired (invalid_grant)":      r"invalid_grant",
        "A - token.json missing":                         r"token\.json not found",
        "B - Google OAuth client invalid (invalid_client)": r"invalid_client",
        "C - Discord token rejected":                     r"LoginFailure|Improper token",
        "D - Claude API key rejected (401)":              r"authentication_error|invalid x-api-key|Error code: 401",
        "D - Out of credit / billing":                    r"credit balance|billing",
    }
    found = False
    for label, pat in checks.items():
        hits = [m.start() for m in re.finditer(pat, text)]
        if hits:
            found = True
            last_line = text[: hits[-1]].count("\n") + 1
            all_lines = text.splitlines()
            i = last_line - 1
            while i > 0 and not re.match(r"\d{4}-\d\d-\d\d", all_lines[i]):
                i -= 1  # tracebacks carry no timestamp -> walk back to the nearest timestamped line
            stamp = all_lines[i][:19]
            print(f"FOUND  {label}  ({len(hits)} times, most recent at {stamp})")
    if not found:
        print("No credential errors found in the log.")
    else:
        print("\nNote: the log keeps OLD errors too. Compare 'most recent at' with the current time;")
        print("if it predates your last reset, you can ignore it.")
    print("\n--- last 5 log lines ---")
    print("\n".join(text.splitlines()[-5:]))

tok = ROOT / "data" / "token.json"
if tok.exists():
    print(f"\ntoken.json was last written {(time.time() - tok.stat().st_mtime) / 86400:.1f} days ago")

> If the bot **answers DMs but stays silent in a channel**, that is not a credential problem. See `01_ENVIRONMENT_SETUP.ipynb`, section 3.4 (channel permissions / channel ID / @mention).

---
# A. Reset the Google Calendar token (`invalid_grant`)

**Most common cause:** the OAuth app is in *Testing* mode, so Google revokes the refresh token after 7 days. The token also dies if you change your Google password or revoke access at https://myaccount.google.com/permissions.

## A1. Prevent a repeat: Publish the app (do this before logging in again)

Google Cloud Console → **Google Auth Platform → Audience → Publish app** (switch to *In production*).
Only tokens created **after** this step stop expiring after 7 days. The app still shows an "unverified" warning; ignore it for personal use.

## A2. Delete the old token

```powershell
cd C:\Calendar_Agent
docker compose down
del data\token.json
```

## A3. Log in again - option 1: through Docker

```powershell
docker compose run --rm -p 8765:8765 calendar-bot python auth.py
```

Open the `https://accounts.google.com/...` URL the terminal prints, in a browser on the **same Windows machine**. Sign in with the Gmail that owns the calendar → **Allow**. The terminal prints `✅ Saved data/token.json`.

> Do not add `--service-ports` (it conflicts with `-p`; Docker says `--service-ports and --publish are incompatible`).

## A3. Log in again - option 2: run directly on Windows (more reliable)

Use this when option 1 ends with `WSGITimeoutError: Timed out waiting for response from authorization server`.

```powershell
cd C:\Calendar_Agent
py -m pip install google-auth-oauthlib
$env:GOOGLE_CREDENTIALS_FILE="data/credentials.json"
$env:GOOGLE_TOKEN_FILE="data/token.json"
py app\auth.py
```

The browser opens by itself. After you click Allow, `data\token.json` is created.

**If both options time out**, check in this order:

1. `netstat -ano | findstr 8765` should print nothing (port 8765 is free).
2. You finished every Google screen quickly (including **Advanced → Go to … (unsafe)**).
3. The Gmail you signed in with is listed under **Test users** (if the app is still in Testing).
4. Windows Firewall / antivirus is not blocking Python or Docker.

## A4. Restart the bot

```powershell
docker compose up -d
docker compose logs -f
```

In [ ]:
from pathlib import Path

# Locate the project root (the folder containing docker-compose.yml),
# whether this notebook is opened from docs/ or from the project root.
ROOT = Path.cwd()
if not (ROOT / "docker-compose.yml").exists() and (ROOT.parent / "docker-compose.yml").exists():
    ROOT = ROOT.parent
print("Project folder:", ROOT)
print("docker-compose.yml found:", (ROOT / "docker-compose.yml").exists())

import json, time

tok = ROOT / "data" / "token.json"
if not tok.exists():
    print("data/token.json does not exist yet -> redo A3")
else:
    d = json.loads(tok.read_text(encoding="utf-8"))
    age_min = (time.time() - tok.stat().st_mtime) / 60
    print(f"token.json written {age_min:.0f} minutes ago")
    print("has refresh_token:", bool(d.get("refresh_token")))
    print("scopes           :", d.get("scopes"))
    print("\nVerdict:", "OK, restart the bot and try a message." if d.get("refresh_token") and age_min < 60
          else "Old token or no refresh_token -> delete it and redo A2-A3.")

**Real-world check:** DM the bot `Am I free today`. Expected: a list of events or "no events", not "auth error". The log should show `Function get_events succeeded` and **no** `RefreshError`.

---
# B. Replace `credentials.json` (OAuth client)

Needed when the file is lost, the `client_secret` leaked, or the client was deleted.

1. Google Cloud Console → **APIs & Services → Credentials** (or *Google Auth Platform → Clients*).
2. Open the old client → reset/delete its secret if it leaked, or **+ Create credentials → OAuth client ID** → type **Desktop app** → Create → **Download JSON**.
3. Save it over `C:\Calendar_Agent\data\credentials.json`.
4. The client changed, so the old token is invalid: redo **A2 → A4**.

In [ ]:
from pathlib import Path

# Locate the project root (the folder containing docker-compose.yml),
# whether this notebook is opened from docs/ or from the project root.
ROOT = Path.cwd()
if not (ROOT / "docker-compose.yml").exists() and (ROOT.parent / "docker-compose.yml").exists():
    ROOT = ROOT.parent
print("Project folder:", ROOT)
print("docker-compose.yml found:", (ROOT / "docker-compose.yml").exists())

import json

cred = ROOT / "data" / "credentials.json"
if not cred.exists():
    print("MISSING data/credentials.json")
else:
    d = json.loads(cred.read_text(encoding="utf-8"))
    kind = next(iter(d), None)
    body = d.get(kind, {})
    print("client type :", kind, "(correct)" if kind == "installed" else "(WRONG: must be a Desktop app -> top-level key 'installed')")
    print("client_id   :", (body.get("client_id") or "")[:12] + "...")
    print("has secret  :", bool(body.get("client_secret")))

---
# C. Reset the Discord bot token

Needed when `.env` or a screenshot with the token leaked, or the bot reports `LoginFailure`.

1. https://discord.com/developers/applications → open the `Calendar` app → **Bot** → **Reset Token** → copy the new token (shown once).
2. Open `.env` and replace the value of `DISCORD_BOT_TOKEN=`.
3. Apply it:

```powershell
cd C:\Calendar_Agent
docker compose up -d --build
docker compose logs -f
```

You should see `Logged in as Calendar#XXXX ... Calendar bot ready.`

Note: if **Message Content Intent** is off (Bot → Privileged Gateway Intents), the bot logs in but cannot read message text. Check that too.

---
# D. Replace the Claude API key

Needed when the key leaked, was revoked (`401`), or the credit ran out.

1. https://console.anthropic.com → **API Keys** → **Create Key** (name `calendar-agent`) → copy it. Delete the old key if you suspect it leaked.
2. Open `.env` and replace the value of `ANTHROPIC_API_KEY=`.
3. If the bot says "API credits exhausted or billing issue": **Billing** → add credit.
4. Apply it:

```powershell
cd C:\Calendar_Agent
docker compose up -d --build
```

---
# E. Verify after any reset

Run the cell below for an overall status (no secrets are printed).

In [ ]:
from pathlib import Path

# Locate the project root (the folder containing docker-compose.yml),
# whether this notebook is opened from docs/ or from the project root.
ROOT = Path.cwd()
if not (ROOT / "docker-compose.yml").exists() and (ROOT.parent / "docker-compose.yml").exists():
    ROOT = ROOT.parent
print("Project folder:", ROOT)
print("docker-compose.yml found:", (ROOT / "docker-compose.yml").exists())

import time

env = {}
p = ROOT / ".env"
if p.exists():
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            env[k.strip()] = v.strip()

print("=== .env ===")
for k in ["DISCORD_BOT_TOKEN", "ALLOWED_DISCORD_USER_IDS", "ANTHROPIC_API_KEY"]:
    print(f"{'OK   ' if env.get(k) else 'EMPTY'}  {k}")
print("Note: DISCORD_CHANNEL_IDS =", env.get("DISCORD_CHANNEL_IDS") or "(empty)")

print("\n=== Google ===")
for f in ["credentials.json", "token.json"]:
    fp = ROOT / "data" / f
    print(("OK       " if fp.exists() else "MISSING  ") + "data/" + f)

print("\n=== Bot ===")
hb = ROOT / "logs" / "heartbeat"
if hb.exists():
    age = time.time() - hb.stat().st_mtime
    print(f"heartbeat {age:.0f}s ago ->", "running" if age < 180 else "may have stopped")
else:
    print("no heartbeat yet")

Then test in Discord:

1. DM: `Am I free today` → must return calendar content.
2. Channel: `@Calendar Am I free today?` → must reply (no @ needed if the channel is in `DISCORD_CHANNEL_IDS`).
3. If the bot answers twice, two containers are running: `docker ps`, then `docker stop $(docker ps -aq)`, `docker rm $(docker ps -aq)`, `docker compose up -d`.

---
# F. If a credential leaked or was committed to GitHub

Order matters: **rotate the secrets first, clean Git second.** A leaked secret is compromised; deleting it from the repo is not enough.

1. Rotate everything that leaked: Discord (section C), Claude (section D), Google client (section B) and token (section A).
2. Check whether Git is tracking any secret file (cell below).
3. If it is, stop tracking it but keep the local file:

```powershell
git rm --cached .env data/credentials.json data/token.json
git commit -m "Stop tracking secrets"
git push
```

4. If it was already pushed, the history still contains the secret. Remove it from history with `git filter-repo` or BFG, and **still rotate the secrets** as in step 1.

In [ ]:
from pathlib import Path

# Locate the project root (the folder containing docker-compose.yml),
# whether this notebook is opened from docs/ or from the project root.
ROOT = Path.cwd()
if not (ROOT / "docker-compose.yml").exists() and (ROOT.parent / "docker-compose.yml").exists():
    ROOT = ROOT.parent
print("Project folder:", ROOT)
print("docker-compose.yml found:", (ROOT / "docker-compose.yml").exists())

import subprocess

def git(*args):
    r = subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True)
    return r.stdout.strip()

tracked = git("ls-files").splitlines()
risky = [f for f in tracked if f == ".env" or f.startswith("data/") or f.startswith("logs/")
         or f.endswith("token.json") or f.endswith("credentials.json")]
print("Secret files tracked by Git:", risky if risky else "none (good)")

ign = (ROOT / ".gitignore").read_text(encoding="utf-8") if (ROOT / ".gitignore").exists() else ""
for need in [".env", "credentials.json", "token.json", "data/", "logs/"]:
    print(("OK       " if need in ign else "MISSING  ") + f".gitignore contains {need}")

---
# G. Quick lookup: errors you may see while resetting

| Message | Meaning | Fix |
|---|---|---|
| `invalid_grant: Bad Request` | The Google refresh token is dead | Section A |
| `WSGITimeoutError: Timed out waiting for response` | Browser could not reach `localhost:8765`, or Allow was not completed | A3 option 2 (run directly on Windows) |
| `--service-ports and --publish are incompatible` | Both flags used together | Use only `-p 8765:8765` |
| `403 access_denied` at login | Gmail not in Test users (app still in Testing) | Add the Gmail to Test users, or Publish the app |
| `403 accessNotConfigured` | Google Calendar API not enabled | Enable it in the API Library |
| `invalid_client` | Wrong `credentials.json`, or the client was deleted | Section B |
| `LoginFailure` / `Improper token` | Wrong Discord token | Section C |
| `401` / `credit balance` | Wrong Claude key, or no credit left | Section D |
| DM works, channel silent | Not a credential issue: channel permissions / wrong ID / no @mention | `01_ENVIRONMENT_SETUP.ipynb`, section 3.4 |